In [13]:
import pandas as pd
import numpy as np

from pymongo import MongoClient
from dotenv import load_dotenv
import os

In [14]:
from pymongo import MongoClient
import pandas as pd

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["aqi_predictor"]

collection = db["aqi_features"]

cursor = (
    collection
    .find({"city": "karachi"})
    .sort("timestamp", 1)
)

df = pd.DataFrame(list(cursor))

In [16]:
print(df["timestamp"].min())
print(df["timestamp"].max())
print(len(df))

2025-08-15T11:00:00+00:00
2026-08-15T15:09:40+00:00
8498


In [17]:
df["city"].value_counts()

city
karachi    8498
Name: count, dtype: int64

In [19]:
df[
    df["timestamp"].str.contains("2025-03-04")
][["timestamp", "aqi", "pm25"]].head(20)

,timestamp,aqi,pm25


In [20]:
df.sort_values("timestamp").head()

,_id,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,month,no2,o3,pm10,pm25,pressure,so2,temperature,wind_speed
0,6a803c9ea2c1b4faf9cc3338,2025-08-15T11:00:00+00:00,karachi,60,0.0,80.73,15,4,11,63.0,8,0.05,38.67,54.27,16.60,998.1,0.15,31.5,13.2
1,6a803c9ea2c1b4faf9cc3339,2025-08-15T12:00:00+00:00,karachi,62,2.0,80.36,15,4,12,72.0,8,0.06,38.42,55.99,17.14,998.5,0.16,30.3,12.7
2,6a803c9ea2c1b4faf9cc333a,2025-08-15T13:00:00+00:00,karachi,63,1.0,80.13,15,4,13,71.0,8,0.08,38.36,58.14,17.65,999.1,0.16,30.2,9.7
3,6a803c9fa2c1b4faf9cc333b,2025-08-15T14:00:00+00:00,karachi,63,0.0,79.89,15,4,14,75.0,8,0.10,38.20,60.32,18.02,999.5,0.17,29.5,8.5
4,6a803c9fa2c1b4faf9cc333c,2025-08-15T15:00:00+00:00,karachi,64,1.0,79.53,15,4,15,79.0,8,0.10,38.08,62.35,18.19,1000.1,0.17,29.0,8.3


In [21]:
df = df[df["timestamp"] != "2025-03-04T16:00:00+05:00"]

In [22]:
print(df["timestamp"].min())

2025-08-15T11:00:00+00:00


In [23]:
df = df[df["timestamp"] != "2025-03-04T16:00:00+05:00"]

In [24]:
df.sort_values("timestamp")[["timestamp","aqi","pm25"]].head(20)

,timestamp,aqi,pm25
0,2025-08-15T11:00:00+00:00,60,16.60
1,2025-08-15T12:00:00+00:00,62,17.14
2,2025-08-15T13:00:00+00:00,63,17.65
3,2025-08-15T14:00:00+00:00,63,18.02
4,2025-08-15T15:00:00+00:00,64,18.19
5,2025-08-15T16:00:00+00:00,64,18.10
6,2025-08-15T17:00:00+00:00,63,17.65
7,2025-08-15T18:00:00+00:00,61,16.98
8,2025-08-15T19:00:00+00:00,59,16.12
9,2025-08-15T20:00:00+00:00,58,15.36


In [25]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [31]:
from src.prediction.build_prediction_features import (
    build_prediction_features,
)

features = build_prediction_features(df)

features.head()

,_id,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,...,aqi_lag_6,aqi_lag_12,aqi_lag_24,pm25_lag_1,pm25_lag_6,pm25_lag_24,aqi_roll_mean_6,aqi_roll_mean_12,aqi_roll_mean_24,aqi_roll_std_24
8495,6a8042a3a2c1b4faf9cc585c,2026-08-15T10:00:00+00:00,karachi,61,-1.0,66.78,15,5,10,73.0,...,63.0,63.0,60.0,17.11,17.86,16.52,62.833333,62.666667,61.708333,1.334465


In [32]:
from src.prediction.predictor import predict

prediction = predict(features)

prediction

{'current_aqi': 61.0, 'day_1': 61.45, 'day_2': 56.45, 'day_3': 59.55}

In [33]:
latest = df.iloc[-1]

print("Current AQI:", latest["aqi"])

Current AQI: 59


In [34]:
from src.features.fetch_openweather import (
    fetch_current_pollution
)

print(fetch_current_pollution("karachi"))

{'main': {'aqi': 3}, 'components': {'co': 67.78, 'no': 0, 'no2': 0.08, 'o3': 46.49, 'so2': 0.48, 'pm2_5': 19.24, 'pm10': 90.29, 'nh3': 0}, 'dt': 1786903748}


In [11]:
from datetime import datetime

data = fetch_current_pollution("karachi")

print(
    datetime.utcfromtimestamp(data["dt"])
)

2026-08-15 13:08:55
